# Author an archetype from materials

Use the installed development revision of `epsm_data` with Python 3.11+. No Core, database, network or engine is needed; no files are written. All numerical values below are illustrative assumptions. Conductivity uses W/(m K), density kg/m³, heat capacity J/(kg K), thickness metres and absorption/fractions dimensionless. The example checks composition and portable exchange, not measured-building accuracy.

In [ ]:
from epsm_data import (
    Archetype, BuildingElement, Component, ConstructionSet, Element,
    EnvelopePerformance, Layer, LayerMaterial, LayerSlot, Material,
    OperatingProfile, Roughness, ShearingLayer,
    archetype_from_dict, archetype_to_dict,
)

material = Material(
    name="Illustrative insulation", json_id="example.insulation",
    boverket_id=None, material_type="illustrative",
    roughness=Roughness.MEDIUM_ROUGH, density=35, specific_heat=840,
    conductivity=0.04, thermal_absorptance=0.9, solar_absorptance=0.6,
    visible_absorptance=0.6,
)
assert material.embodied_carbon is None

Composition is explicit: material → layer → component → element → construction set. Layer slots are ordered outside to inside, and parallel material fractions sum to one. An unknown construction year remains `None`. Unknown carbon is not zero.

In [ ]:
role = BuildingElement.EXT_WALL
layer = Layer("Insulation", "example.layer", 0.2, (LayerMaterial(material, 1.0, 0.2),))
component = Component("Wall layer", "example", role, (LayerSlot(layer, 0),), ShearingLayer.STRUCTURE)
element = Element("Custom wall", "example", None, (component,), role)
construction = ConstructionSet(elements={role: element})
archetype = Archetype(
    "example.custom", "Custom research building",
    building_typology="research:custom", construction_set=construction,
    source_references=("example:declared-assumptions",),
)
assert archetype.construction_year is None
assert archetype.total_elements == 1
assert "hvac_template" in archetype.missing_sections
print(archetype)

Profiles and envelope assumptions are independently reusable. This operating profile is deliberately partial; Core must validate and complete it later. The 0.2 ACH rate explicitly refers to conditioned volume. No installed system, project calendar, reference area or site condition is inferred.

In [ ]:
profile = OperatingProfile(
    "example.partial-use",
    {"building_assumptions": {"use": {"occupant_density_per_m2": 0.04}}},
    notes=("Partial illustrative operation; not simulation ready.",),
)
authored = archetype.with_changes(
    operating_profile=profile,
    envelope_performance=EnvelopePerformance(
        infiltration_ach=0.2, infiltration_reference_basis="conditioned_volume",
        source_references=("example:declared-assumptions",),
    ),
)
assert archetype.operating_profile is None
assert authored.construction_set == archetype.construction_set
assert "hvac_template" in authored.summary()["missing_sections"]

Format 4 stores the full composition and optional sections. Round-tripping needs no catalogue lookup. Immutable editing preserves the original and its material identity.

In [ ]:
import json

record = archetype_to_dict(authored)
assert record["format_version"] == 4
restored = archetype_from_dict(json.loads(json.dumps(record)))
assert restored == authored
restored_material = restored.construction_set.elements[role].components[0].layers[0].layer.materials[0].material
assert restored_material.uuid == material.uuid
edited = restored.with_changes(notes=("Independent edited copy.",))
assert edited.notes != authored.notes
print(restored)